# Data Exploration Notebook

This notebook explores the data pipeline for the CNN Stock Market Prediction project.

## Contents
1. Fetch sample stock data
2. Visualize raw OHLCV data
3. Show normalization effect
4. Display sliding windows
5. Show label distribution
6. Test DataLoader iteration

In [ ]:
# Add parent directory to path for imports
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Imports successful!')

## 1. Fetch Sample Stock Data

We'll fetch historical OHLCV data for Apple (AAPL) as a sample.

In [ ]:
from src.data.fetcher import fetch_stock_data, fetch_sp500_tickers

# Fetch AAPL data from 2015 to present
ticker = 'AAPL'
start_date = '2015-01-01'
end_date = '2024-01-01'

df = fetch_stock_data(ticker, start_date, end_date)
print(f'Fetched {len(df)} rows for {ticker}')
print(f'Date range: {df.index[0]} to {df.index[-1]}')
print(f'\nColumns: {list(df.columns)}')
df.head()

In [ ]:
# Check S&P 500 tickers
sp500_tickers = fetch_sp500_tickers()
print(f'S&P 500 contains {len(sp500_tickers)} stocks')
print(f'First 10: {sp500_tickers[:10]}')

## 2. Visualize Raw OHLCV Data

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Price plot
ax1 = axes[0]
ax1.plot(df.index, df['Close'], label='Close', linewidth=1)
ax1.fill_between(df.index, df['Low'], df['High'], alpha=0.3, label='High-Low Range')
ax1.set_ylabel('Price ($)')
ax1.set_title(f'{ticker} Stock Price (2015-2024)')
ax1.legend()

# Volume plot
ax2 = axes[1]
ax2.bar(df.index, df['Volume'] / 1e6, alpha=0.7, width=1)
ax2.set_ylabel('Volume (M)')
ax2.set_xlabel('Date')

plt.tight_layout()
plt.show()

In [ ]:
# Data statistics
df.describe()

## 3. Show Normalization Effect

We normalize each feature to [0, 1] using min-max scaling.

In [ ]:
from src.data.preprocessor import normalize

df_normalized = normalize(df, method='minmax')

print('Original data range:')
print(df.min())
print(df.max())

print('\nNormalized data range:')
print(df_normalized.min())
print(df_normalized.max())

In [ ]:
# Compare original vs normalized
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Original
axes[0].plot(df.index[-200:], df['Close'].iloc[-200:], label='Close')
axes[0].plot(df.index[-200:], df['Volume'].iloc[-200:] / df['Volume'].max(), label='Volume (scaled)')
axes[0].set_title('Original Data (Last 200 Days)')
axes[0].legend()

# Normalized
for col in df_normalized.columns:
    axes[1].plot(df_normalized.index[-200:], df_normalized[col].iloc[-200:], label=col, alpha=0.7)
axes[1].set_title('Normalized Data (Last 200 Days)')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 4. Display Sliding Windows

Create sliding windows of 256 days with a 5-day prediction horizon.

In [ ]:
from src.data.preprocessor import create_sliding_windows

window_size = 256
horizon = 5

X, y = create_sliding_windows(df, window_size=window_size, horizon=horizon)

print(f'X shape: {X.shape}')  # [N, 256, 5]
print(f'y shape: {y.shape}')  # [N,]
print(f'\nData type X: {X.dtype}')
print(f'Data type y: {y.dtype}')
print(f'\nLabel distribution: {y.mean():.2%} bullish, {1-y.mean():.2%} bearish')

In [ ]:
# Visualize a sample window
sample_idx = 100
sample_window = X[sample_idx]
sample_label = y[sample_idx]

fig, ax = plt.subplots(figsize=(12, 5))

feature_names = ['Open', 'High', 'Low', 'Close', 'Volume']
for i, name in enumerate(feature_names[:4]):  # Skip volume for clarity
    ax.plot(sample_window[:, i], label=name, alpha=0.8)

label_text = 'BULLISH (↑)' if sample_label == 1 else 'BEARISH (↓)'
ax.set_title(f'Sample Window #{sample_idx} - Next {horizon} Days: {label_text}')
ax.set_xlabel('Day in Window')
ax.set_ylabel('Normalized Value')
ax.legend()
ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Visualize multiple windows overlaid
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bullish samples
bullish_indices = np.where(y == 1)[0][:10]
for idx in bullish_indices:
    axes[0].plot(X[idx, :, 3], alpha=0.5)  # Close price
axes[0].set_title('10 Bullish Windows (Close Price)')
axes[0].set_xlabel('Day')
axes[0].set_ylabel('Normalized Close')

# Bearish samples
bearish_indices = np.where(y == 0)[0][:10]
for idx in bearish_indices:
    axes[1].plot(X[idx, :, 3], alpha=0.5)  # Close price
axes[1].set_title('10 Bearish Windows (Close Price)')
axes[1].set_xlabel('Day')
axes[1].set_ylabel('Normalized Close')

plt.tight_layout()
plt.show()

## 5. Show Label Distribution

In [ ]:
from src.data.preprocessor import train_val_test_split

# Split data
X_train, y_train, X_val, y_val, X_test, y_test = train_val_test_split(
    X, y, val_ratio=0.15, test_ratio=0.15
)

print(f'Train set: {len(X_train)} samples')
print(f'Validation set: {len(X_val)} samples')
print(f'Test set: {len(X_test)} samples')

In [ ]:
# Label distribution per split
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

splits = [
    ('Train', y_train),
    ('Validation', y_val),
    ('Test', y_test)
]

for ax, (name, labels) in zip(axes, splits):
    bullish = (labels == 1).sum()
    bearish = (labels == 0).sum()
    ax.bar(['Bearish', 'Bullish'], [bearish, bullish], color=['red', 'green'], alpha=0.7)
    ax.set_title(f'{name} Set\n({len(labels)} samples)')
    ax.set_ylabel('Count')
    for i, v in enumerate([bearish, bullish]):
        ax.text(i, v + 5, f'{v}\n({v/len(labels):.1%})', ha='center')

plt.tight_layout()
plt.show()

## 6. Test DataLoader Iteration

In [ ]:
from src.data.dataset import create_dataloaders

# Create DataLoaders
batch_size = 32
train_loader, val_loader = create_dataloaders(
    X_train, y_train, X_val, y_val, batch_size=batch_size
)

print(f'Train DataLoader: {len(train_loader)} batches')
print(f'Validation DataLoader: {len(val_loader)} batches')

In [ ]:
# Get a sample batch
X_batch, y_batch = next(iter(train_loader))

print(f'Batch X shape: {X_batch.shape}')  # [batch_size, 256, 5]
print(f'Batch y shape: {y_batch.shape}')  # [batch_size,]
print(f'X dtype: {X_batch.dtype}')
print(f'y dtype: {y_batch.dtype}')
print(f'\nBatch label distribution: {y_batch.float().mean():.2%} bullish')

In [ ]:
# Verify data shapes are correct for the CNN model
print('Expected CNN input shape: [batch, window_size, channels]')
print(f'Actual batch shape: {X_batch.shape}')
print(f'\nWindow size: {X_batch.shape[1]} (expected 256)')
print(f'Channels: {X_batch.shape[2]} (expected 5 for OHLCV)')

assert X_batch.shape[1] == 256, 'Window size mismatch!'
assert X_batch.shape[2] == 5, 'Channel count mismatch!'
print('\n✓ Data shapes are correct for CNN input!')

## Summary

The data pipeline is working correctly:
- ✓ Can fetch stock data from Yahoo Finance
- ✓ Data is normalized to [0, 1] range
- ✓ Sliding windows have correct shape [N, 256, 5]
- ✓ Labels are binary (0=bearish, 1=bullish)
- ✓ Train/val/test split preserves chronological order
- ✓ DataLoader provides correct batch shapes for CNN